In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import yaml

cwd = Path.cwd()
CONFIG_PATH = cwd.parent / "config" / "config.yaml"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

ROOT = Path(cfg["paths"]["htem_filtered_data_root"]).resolve()
assert ROOT.exists(), f"ROOT does not exist: {ROOT}"

In [2]:
def read_json(p: Path) -> dict:
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)

def angle_to_suffix(angle: float) -> str:
    """
    19.05 -> '1905' (angle*100 rounded).
    """
    if angle is None:
        return None
    try:
        return f"{int(round(float(angle) * 100)):04d}"
    except Exception:
        return None

def is_deposition_key(k: str) -> bool:
    return isinstance(k, str) and k.startswith("deposition") and k not in ["deposition_compounds", "deposition_power", "deposition_gases", "deposition_gas_flow_sccm", "deposition_substrate_material"]

In [3]:
library_dirs = [p for p in ROOT.iterdir() if p.is_dir()]
print("Library dirs found:", len(library_dirs))

libraries = []
samples_by_id = {}

# Track missing situations
missing_samples_folder = 0
missing_library_json = 0

for lib_dir in library_dirs:
    # Find library json: typically one json in the lib root that's not the samples folder
    lib_json_candidates = [p for p in lib_dir.glob("*.json") if p.is_file()]
    if not lib_json_candidates:
        missing_library_json += 1
        continue

    # If multiple, pick the one that looks most like a library file (fallback: first)
    lib_json_path = None
    for p in lib_json_candidates:
        name = p.name.lower()
        if "library" in name or name == f"{lib_dir.name}.json":
            lib_json_path = p
            break
    lib_json_path = lib_json_path or lib_json_candidates[0]

    lib = read_json(lib_json_path)
    libraries.append(lib)

    samples_dir = lib_dir / "samples"
    if not samples_dir.exists():
        missing_samples_folder += 1
        continue

    for sample_path in samples_dir.glob("*.json"):
        s = read_json(sample_path)
        sid = s.get("id")
        if sid is None:
            # If your sample JSONs are named by id, you could derive it; leaving as skip to avoid silent wrong joins
            continue
        samples_by_id[sid] = s

print("Loaded libraries:", len(libraries))
print("Loaded samples:", len(samples_by_id))
print("Libraries missing JSON:", missing_library_json)
print("Libraries missing samples folder:", missing_samples_folder)


Library dirs found: 220
Loaded libraries: 220
Loaded samples: 9680
Libraries missing JSON: 0
Libraries missing samples folder: 0


In [4]:
all_angle_suffixes = set()
all_xrf_compounds = set()
all_deposition_compounds = set()
all_deposition_gases = set()
all_deposition_substrate_materials = set()

deposition_gases_lookup = {
    'ARGON':'Ar', 
    'Ar':'Ar', 
    'Argon':'Ar', 
    'HYDROGEN':'H2', 
    'N2':'N2', 
    'NITROGEN':'N2', 
    'Nitrogen':'N2', 
    'OXYGEN': 'O2'
}

deposition_substrate_materials_lookup = {
    'EXG':'ExG', 
    'Eagle 2k':'Eagle 2k', 
    'ExG':'ExG', 
    'Silicon with 100nm SiO2': 'si_100nm_thermal_oxide', 
    'Silicon with 100nm thermal oxide': 'si_100nm_thermal_oxide'
}

for l in libraries:
    comps = l.get("deposition_compounds") or []
    if comps is None: 
        continue

    for c in comps:
        if c is not None:
            all_deposition_compounds.add(c)

    gases = l.get("deposition_gases") or []
    if gases is None:
        continue

    for g in gases:
        if g is not None and g != "None":
            all_deposition_gases.add(deposition_gases_lookup[g])

    mat = l.get("deposition_substrate_material") or None
    if mat is not None:
        all_deposition_substrate_materials.add(deposition_substrate_materials_lookup[mat])

missing_xrd_samples = 0

for sid, s in samples_by_id.items():
    angles = s.get("xrd_angle")
    if not angles:
        missing_xrd_samples += 1
    else:
        for a in angles:
            suf = angle_to_suffix(a)
            if suf is not None:
                all_angle_suffixes.add(suf)

    comps = s.get("xrf_compounds") or []
    for c in comps:
        if c is not None:
            all_xrf_compounds.add(c)

all_angle_suffixes = sorted(all_angle_suffixes, key=lambda x: int(x))
all_xrf_compounds = sorted(all_xrf_compounds)
all_deposition_compounds = sorted(all_deposition_compounds)
all_deposition_gases = sorted(all_deposition_gases)
all_deposition_substrate_materials = sorted(all_deposition_substrate_materials)

# xrd_bg_cols = [f"xrd_background_{suf}" for suf in all_angle_suffixes]
# xrd_int_cols = [f"xrd_intensity_{suf}" for suf in all_angle_suffixes]
# xrf_has_cols = [f"xrf_has_{c}" for c in all_xrf_compounds]
# xrf_pct_cols = [f"xrf_pct_{c}" for c in all_xrf_compounds]

print("Unique XRD angles:", len(all_angle_suffixes))
print("Unique XRF compounds:", len(all_xrf_compounds))
print("Unique Deposition compounds:", len(all_deposition_compounds))
print("Unique Deposition gases:", len(all_deposition_gases))
print("Unique Deposition substrate materials:", len(all_deposition_substrate_materials))
print("Samples missing XRD:", missing_xrd_samples)

Unique XRD angles: 661
Unique XRF compounds: 28
Unique Deposition compounds: 26
Unique Deposition gases: 4
Unique Deposition substrate materials: 3
Samples missing XRD: 0


In [5]:
print(all_xrf_compounds)
print(all_deposition_compounds)
print(all_deposition_gases)
print(all_deposition_substrate_materials)

['Co', 'CoO', 'Cr', 'CrO', 'Cu2O', 'Cu3N', 'Fe', 'Ga', 'Ge', 'InO', 'MnO', 'Mo', 'NiO', 'Se2O', 'Sn', 'SnO', 'SnO2', 'Ta', 'Ta3N5', 'TaO', 'Ti', 'TiO2', 'V', 'W', 'Zn', 'Zn2O2', 'Zn3N', 'ZnO']
['Co', 'CoO', 'Cr', 'Cr2O3', 'Cu', 'Cu2O', 'Ga2O3', 'Ge', 'In2O3', 'Mg', 'MgO', 'Mn', 'Mn2O3', 'MnO', 'Mo', 'N', 'N2', 'NiO', 'Sn', 'SnO2', 'Ta', 'TiO2', 'V', 'W', 'Zn', 'ZnO']
['Ar', 'H2', 'N2', 'O2']
['Eagle 2k', 'ExG', 'si_100nm_thermal_oxide']


In [6]:
def build_rows(libraries, samples_by_id, all_angle_suffixes, all_xrf_compounds, all_deposition_compounds, all_deposition_gases, all_deposition_substrate_materials):
    rows = []
    all_angle_set = set(all_angle_suffixes)

    for lib in libraries:
        lib_id = lib.get("id")
        sample_ids = lib.get("sample_ids") or []

        # Library fields
        base = {"library_id": lib_id}
        for k, v in lib.items():
            if is_deposition_key(k):
                # if not (isinstance(v, list)) and v is not None:
                #     base[k] = v
                base[k] = np.nan
                if v is not None:
                    base[k] = v
       
        # Initialize Deposition Compound Power to 0
        for c in all_deposition_compounds:
            base[f"deposition_compound_{c}_power"] = 0

        d_comps = lib.get("deposition_compounds") or []
        d_comps_power = lib.get("deposition_power") or []

        if (len(d_comps) != len(d_comps_power)):
            print(f"Error! Library {lib.get('id')} mismatch deposition compounds/power")
        else:
            for idx, c in enumerate(d_comps):
                if c is not None:
                    base[f"deposition_compound_{c}_power"] = d_comps_power[idx]

        # Initialize Deposition gases SCCM (Standard Cubic Centimeters per Minute)
        for g in all_deposition_gases:
            base[f"deposition_gas_{g}_sccm"] = 0

        d_gases = lib.get("deposition_gases") or []
        d_gases_sccm = lib.get("deposition_gas_flow_sccm") or []

        if (len(d_gases) != len(d_gases_sccm)):
            print(f"Error! Library {lib.get('id')} mismatch deposition gases/sccm")
        else:
            for idx, g in enumerate(d_gases):
                if g is not None and g != "None":
                    base[f"deposition_gas_{deposition_gases_lookup[g]}_sccm"] = d_gases_sccm[idx]

        # Initialize Deposition Substrate Materials to 0
        for mat in all_deposition_substrate_materials:
            base[f"deposition_substrate_material_{mat}"] = 0

        material = lib.get("deposition_substrate_material") or None
        if material is not None:
            base[f"deposition_substrate_material_{deposition_substrate_materials_lookup[material]}"] = 1
        
        # Reference angles list for within-library consistency check
        ref_angles = [19, 19.05, 19.1, 19.15, 19.2, 19.25, 19.3, 19.35, 19.4, 19.45, 19.5, 19.55, 19.6, 19.65, 19.7, 19.75, 19.8, 19.85, 19.9, 19.95, 20, 20.05, 20.1, 20.15, 20.2, 20.25, 20.3, 20.35, 20.4, 20.45, 20.5, 20.55, 20.6, 20.65, 20.7, 20.75, 20.8, 20.85, 20.9, 20.95, 21, 21.05, 21.1, 21.15, 21.2, 21.25, 21.3, 21.35, 21.4, 21.45, 21.5, 21.55, 21.6, 21.65, 21.7, 21.75, 21.8, 21.85, 21.9, 21.95, 22, 22.05, 22.1, 22.15, 22.2, 22.25, 22.3, 22.35, 22.4, 22.45, 22.5, 22.55, 22.6, 22.65, 22.7, 22.75, 22.8, 22.85, 22.9, 22.95, 23, 23.05, 23.1, 23.15, 23.2, 23.25, 23.3, 23.35, 23.4, 23.45, 23.5, 23.55, 23.6, 23.65, 23.7, 23.75, 23.8, 23.85, 23.9, 23.95, 24, 24.05, 24.1, 24.15, 24.2, 24.25, 24.3, 24.35, 24.4, 24.45, 24.5, 24.55, 24.6, 24.65, 24.7, 24.75, 24.8, 24.85, 24.9, 24.95, 25, 25.05, 25.1, 25.15, 25.2, 25.25, 25.3, 25.35, 25.4, 25.45, 25.5, 25.55, 25.6, 25.65, 25.7, 25.75, 25.8, 25.85, 25.9, 25.95, 26, 26.05, 26.1, 26.15, 26.2, 26.25, 26.3, 26.35, 26.4, 26.45, 26.5, 26.55, 26.6, 26.65, 26.7, 26.75, 26.8, 26.85, 26.9, 26.95, 27, 27.05, 27.1, 27.15, 27.2, 27.25, 27.3, 27.35, 27.4, 27.45, 27.5, 27.55, 27.6, 27.65, 27.7, 27.75, 27.8, 27.85, 27.9, 27.95, 28, 28.05, 28.1, 28.15, 28.2, 28.25, 28.3, 28.35, 28.4, 28.45, 28.5, 28.55, 28.6, 28.65, 28.7, 28.75, 28.8, 28.85, 28.9, 28.95, 29, 29.05, 29.1, 29.15, 29.2, 29.25, 29.3, 29.35, 29.4, 29.45, 29.5, 29.55, 29.6, 29.65, 29.7, 29.75, 29.8, 29.85, 29.9, 29.95, 30, 30.05, 30.1, 30.15, 30.2, 30.25, 30.3, 30.35, 30.4, 30.45, 30.5, 30.55, 30.6, 30.65, 30.7, 30.75, 30.8, 30.85, 30.9, 30.95, 31, 31.05, 31.1, 31.15, 31.2, 31.25, 31.3, 31.35, 31.4, 31.45, 31.5, 31.55, 31.6, 31.65, 31.7, 31.75, 31.8, 31.85, 31.9, 31.95, 32, 32.05, 32.1, 32.15, 32.2, 32.25, 32.3, 32.35, 32.4, 32.45, 32.5, 32.55, 32.6, 32.65, 32.7, 32.75, 32.8, 32.85, 32.9, 32.95, 33, 33.05, 33.1, 33.15, 33.2, 33.25, 33.3, 33.35, 33.4, 33.45, 33.5, 33.55, 33.6, 33.65, 33.7, 33.75, 33.8, 33.85, 33.9, 33.95, 34, 34.05, 34.1, 34.15, 34.2, 34.25, 34.3, 34.35, 34.4, 34.45, 34.5, 34.55, 34.6, 34.65, 34.7, 34.75, 34.8, 34.85, 34.9, 34.95, 35, 35.05, 35.1, 35.15, 35.2, 35.25, 35.3, 35.35, 35.4, 35.45, 35.5, 35.55, 35.6, 35.65, 35.7, 35.75, 35.8, 35.85, 35.9, 35.95, 36, 36.05, 36.1, 36.15, 36.2, 36.25, 36.3, 36.35, 36.4, 36.45, 36.5, 36.55, 36.6, 36.65, 36.7, 36.75, 36.8, 36.85, 36.9, 36.95, 37, 37.05, 37.1, 37.15, 37.2, 37.25, 37.3, 37.35, 37.4, 37.45, 37.5, 37.55, 37.6, 37.65, 37.7, 37.75, 37.8, 37.85, 37.9, 37.95, 38, 38.05, 38.1, 38.15, 38.2, 38.25, 38.3, 38.35, 38.4, 38.45, 38.5, 38.55, 38.6, 38.65, 38.7, 38.75, 38.8, 38.85, 38.9, 38.95, 39, 39.05, 39.1, 39.15, 39.2, 39.25, 39.3, 39.35, 39.4, 39.45, 39.5, 39.55, 39.6, 39.65, 39.7, 39.75, 39.8, 39.85, 39.9, 39.95, 40, 40.05, 40.1, 40.15, 40.2, 40.25, 40.3, 40.35, 40.4, 40.45, 40.5, 40.55, 40.6, 40.65, 40.7, 40.75, 40.8, 40.85, 40.9, 40.95, 41, 41.05, 41.1, 41.15, 41.2, 41.25, 41.3, 41.35, 41.4, 41.45, 41.5, 41.55, 41.6, 41.65, 41.7, 41.75, 41.8, 41.85, 41.9, 41.95, 42, 42.05, 42.1, 42.15, 42.2, 42.25, 42.3, 42.35, 42.4, 42.45, 42.5, 42.55, 42.6, 42.65, 42.7, 42.75, 42.8, 42.85, 42.9, 42.95, 43, 43.05, 43.1, 43.15, 43.2, 43.25, 43.3, 43.35, 43.4, 43.45, 43.5, 43.55, 43.6, 43.65, 43.7, 43.75, 43.8, 43.85, 43.9, 43.95, 44, 44.05, 44.1, 44.15, 44.2, 44.25, 44.3, 44.35, 44.4, 44.45, 44.5, 44.55, 44.6, 44.65, 44.7, 44.75, 44.8, 44.85, 44.9, 44.95, 45, 45.05, 45.1, 45.15, 45.2, 45.25, 45.3, 45.35, 45.4, 45.45, 45.5, 45.55, 45.6, 45.65, 45.7, 45.75, 45.8, 45.85, 45.9, 45.95, 46, 46.05, 46.1, 46.15, 46.2, 46.25, 46.3, 46.35, 46.4, 46.45, 46.5, 46.55, 46.6, 46.65, 46.7, 46.75, 46.8, 46.85, 46.9, 46.95, 47, 47.05, 47.1, 47.15, 47.2, 47.25, 47.3, 47.35, 47.4, 47.45, 47.5, 47.55, 47.6, 47.65, 47.7, 47.75, 47.8, 47.85, 47.9, 47.95, 48, 48.05, 48.1, 48.15, 48.2, 48.25, 48.3, 48.35, 48.4, 48.45, 48.5, 48.55, 48.6, 48.65, 48.7, 48.75, 48.8, 48.85, 48.9, 48.95, 49, 49.05, 49.1, 49.15, 49.2, 49.25, 49.3, 49.35, 49.4, 49.45, 49.5, 49.55, 49.6, 49.65, 49.7, 49.75, 49.8, 49.85, 49.9, 49.95, 50, 50.05, 50.1, 50.15, 50.2, 50.25, 50.3, 50.35, 50.4, 50.45, 50.5, 50.55, 50.6, 50.65, 50.7, 50.75, 50.8, 50.85, 50.9, 50.95, 51, 51.05, 51.1, 51.15, 51.2, 51.25, 51.3, 51.35, 51.4, 51.45, 51.5, 51.55, 51.6, 51.65, 51.7, 51.75, 51.8, 51.85, 51.9, 51.95, 52]
        ref_sid = 206906
        printed_mismatch = False

        for sid in sample_ids:
            s = samples_by_id.get(sid)
            
            # row = dict(base)
            # row["sample_id"] = sid

            row = {"sample_id": sid}
            row.update(base)

            # Thickness
            row["thickness"] = np.nan if s is None else s.get("thickness", np.nan)

            # Initialize XRF: pct=0
            for c in all_xrf_compounds:
                # row[f"xrf_has_{c}"] = 0  
                row[f"xrf_pct_{c}"] = 0
            
            # Initialize XRD to NaN
            for suf in all_angle_suffixes:
                row[f"xrd_background_{suf}"] = np.nan
                row[f"xrd_intensity_{suf}"] = np.nan

            if s is None:
                rows.append(row)
                continue

            # ---- XRD flatten ----
            angles = s.get("xrd_angle")
            bg = s.get("xrd_background")
            inten = s.get("xrd_intensity")

            if angles and bg and inten:
                # Angle match check (list equality)
                if list(angles) != ref_angles and not printed_mismatch:
                    print(f"\n[ANGLE MISMATCH] library_id={lib_id} (ref sample_id={ref_sid}, mismatch sample_id={sid})")
                    # print(lib)  # print the library dict as requested
                    printed_mismatch = True

                # Populate columns
                for a, b, i in zip(angles, bg, inten):
                    suf = angle_to_suffix(a)
                    if suf is None:
                        continue
                    if suf in all_angle_set:
                        row[f"xrd_background_{suf}"] = b
                        row[f"xrd_intensity_{suf}"] = i

            # ---- XRF flatten ----
            comps = s.get("xrf_compounds") or []
            concs = s.get("xrf_concentration") or []

            for idx, c in enumerate(comps):
                if c is None:
                    continue
                # row[f"xrf_has_{c}"] = 1
                if idx < len(concs):
                    row[f"xrf_pct_{c}"] = concs[idx]

            rows.append(row)

    return rows

rows = build_rows(libraries, samples_by_id, all_angle_suffixes, all_xrf_compounds, all_deposition_compounds, all_deposition_gases, all_deposition_substrate_materials)
print("Rows:", len(rows))


[ANGLE MISMATCH] library_id=10105 (ref sample_id=206906, mismatch sample_id=322853)

[ANGLE MISMATCH] library_id=12585 (ref sample_id=206906, mismatch sample_id=387214)

[ANGLE MISMATCH] library_id=12606 (ref sample_id=206906, mismatch sample_id=387617)

[ANGLE MISMATCH] library_id=6630 (ref sample_id=206906, mismatch sample_id=207122)

[ANGLE MISMATCH] library_id=6738 (ref sample_id=206906, mismatch sample_id=210185)

[ANGLE MISMATCH] library_id=6995 (ref sample_id=206906, mismatch sample_id=217124)

[ANGLE MISMATCH] library_id=7172 (ref sample_id=206906, mismatch sample_id=221596)

[ANGLE MISMATCH] library_id=8376 (ref sample_id=206906, mismatch sample_id=301108)

[ANGLE MISMATCH] library_id=8405 (ref sample_id=206906, mismatch sample_id=330643)
Rows: 9680


In [7]:
df = pd.DataFrame(rows)
print("df shape:", df.shape)

# Expected rows (if every library truly has 44 sample_ids)
expected = len(libraries) * 44
if df.shape[0] != expected:
    print(f"WARNING: expected {expected} rows, got {df.shape[0]}")
    # Quick diagnostic
    counts = df.groupby("library_id")["sample_id"].count().describe()
    print("Per-library row count summary:\n", counts)
else:
    print("Row count matches len(libraries)*44.")

# ---- SAVE AS CSV ----
SAVE_ROOT = cwd.parent / "datasets"
out_path = SAVE_ROOT / "htem_220library_dataset.csv"
df.to_csv(out_path, index=False)

print("Saved:", out_path)

df.head()

df shape: (9680, 1395)
Row count matches len(libraries)*44.
Saved: C:\Users\danma\Documents\Dan\Projects\Materials Project\helper_files\htem-api-examples\datasets\htem_220library_dataset.csv


,sample_id,library_id,deposition_sample_time_min,deposition_base_pressure_mtorr,deposition_growth_pressure_mtorr,deposition_target_pulses,deposition_rep_rate,deposition_energy,deposition_cycles,deposition_ts_distance,...,xrd_background_5180,xrd_intensity_5180,xrd_background_5185,xrd_intensity_5185,xrd_background_5190,xrd_intensity_5190,xrd_background_5195,xrd_intensity_5195,xrd_background_5200,xrd_intensity_5200
0,322800,10104,60,0.006,10.0,NaN,[None],NaN,NaN,NaN,...,3135.0,3288.0,3132.0,3235.0,3157.0,3162.0,3117.0,3089.0,3100.0,3100.0
1,322802,10104,60,0.006,10.0,NaN,[None],NaN,NaN,NaN,...,3078.0,3060.0,3087.0,3064.0,3058.0,3131.0,3031.0,2937.0,3025.0,3025.0
2,322801,10104,60,0.006,10.0,NaN,[None],NaN,NaN,NaN,...,3161.0,3161.0,3138.0,3194.0,3114.0,3084.0,3092.0,3055.0,3137.0,3137.0
3,322804,10104,60,0.006,10.0,NaN,[None],NaN,NaN,NaN,...,3276.0,3314.0,3274.0,3290.0,3263.0,3231.0,3265.0,3301.0,3211.0,3211.0
4,322803,10104,60,0.006,10.0,NaN,[None],NaN,NaN,NaN,...,3227.0,3246.0,3214.0,3292.0,3201.0,3223.0,3175.0,3132.0,3169.0,3169.0


In [8]:
# 9 Angle mismatches matches the 185 mismatched samples - 176 removed samples = 9 rotated samples, so all the angles are properly standardized